In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv


In [2]:
import pandas as pd
import numpy as np
import nltk
# from nltk.corpus import stopwords
# nltk.download('stopwords')

In [3]:
df = pd.read_csv('/kaggle/input/datasets/organizations/uciml/sms-spam-collection-dataset/spam.csv', encoding = 'latin-1')

In [4]:
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [5]:
drop_columns = ['Unnamed: 2','Unnamed: 3','Unnamed: 4']
df.drop(columns = drop_columns, inplace = True)

In [6]:
df.shape

(5572, 2)

In [7]:
df.columns

Index(['v1', 'v2'], dtype='object')

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   v1      5572 non-null   object
 1   v2      5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [9]:
df.isnull().sum()

v1    0
v2    0
dtype: int64

In [10]:
new_column_name = {'v1': 'Category', 'v2': 'message'}
df.rename(columns = new_column_name, inplace = True)

In [11]:
df.head()

,Category,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [12]:
df['Category'].value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [13]:
#Create message length feature
df['message_length'] = df['message'].apply(len)

In [14]:
#Compare spam vs ham
df.groupby('Category')['message_length'].mean()

Category
ham      71.023627
spam    138.866131
Name: message_length, dtype: float64

In [15]:
#Find longest messages
df.sort_values(by= 'message_length', ascending = False).head()

,Category,message,message_length
1084,ham,For me the love should start with attraction.i...,910
1862,ham,The last thing i ever wanted to do was hurt yo...,790
2433,ham,Indians r poor but India is not a poor country...,632
1578,ham,How to Make a girl Happy? It's not at all diff...,611
2847,ham,Sad story of a Man - Last week was my b'day. M...,588


In [16]:
#print some spam messages
df[df['Category'] == 'spam'].head()

,Category,message,message_length
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155
5,spam,FreeMsg Hey there darling it's been 3 week's n...,148
8,spam,WINNER!! As a valued network customer you have...,158
9,spam,Had your mobile 11 months or more? U R entitle...,154
11,spam,"SIX chances to win CASH! From 100 to 20,000 po...",136


In [17]:
df['original_text'] = df['message']

Preprocessing


In [18]:
df['message'] = df['message'].str.lower()

In [19]:
import re
def remove_punctuation(text):
    return re.sub(r'[^\w\s]','',text)
df['message'] = df['message'].apply(remove_punctuation)

In [20]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stop_words])

df['message'] = df['message'].apply(remove_stopwords)

# print(list(stopwords.words('english'))[:20])

In [21]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()

def stemming(text):
    return " ".join([ps.stem(word) for word in text.split()])

df['message'] = df['message'].apply(stemming)

In [22]:
def preprocess(text):
    text = text.lower()
    text = remove_punctuation(text)
    text = remove_stopwords(text)
    text = stemming(text)
    return text
df['cleaned_message'] = df['original_text'].apply(preprocess)

In [23]:
sample = "WINNER!! You have won a FREE prize. Click now!!!"

print("Original:", sample)
print("Processed:", preprocess(sample))

Original: WINNER!! You have won a FREE prize. Click now!!!
Processed: winner free prize click


In [24]:
df[['original_text', 'cleaned_message']].head()

,original_text,cleaned_message
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf live around though


In [25]:
df[df['Category']=='spam'][['original_text', 'cleaned_message']].head()

,original_text,cleaned_message
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey darl 3 week word back id like fun ...
8,WINNER!! As a valued network customer you have...,winner valu network custom select receivea å90...
9,Had your mobile 11 months or more? U R entitle...,mobil 11 month u r entitl updat latest colour ...
11,"SIX chances to win CASH! From 100 to 20,000 po...",six chanc win cash 100 20000 pound txt csh11 s...


Feature Engineering

In [26]:
#Bag of Words (CountVectorizer)
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

x = cv.fit_transform(df['cleaned_message'])

In [27]:
print(x.shape)

(5572, 8069)


In [28]:
print(cv.get_feature_names_out()[:20])

['008704050406' '0089mi' '0121' '01223585236' '01223585334' '0125698789'
 '02' '020603' '0207' '02070836089' '02072069400' '02073162414'
 '02085076972' '020903' '021' '050703' '0578' '06' '060505' '061104']


In [29]:
#Train First Model- split
from sklearn.model_selection import train_test_split

y = df['Category']
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

In [30]:
#Train Naive Bayes (best for text)
from sklearn.naive_bayes import MultinomialNB
NB = MultinomialNB(alpha = 0.1)
NB.fit(x_train, y_train)

MultinomialNB(alpha=0.1)

In [31]:
y_pred = NB.predict(x_test)

In [32]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy: ", accuracy_score(y_test, y_pred))
print('classfication_report', classification_report(y_test, y_pred))

Accuracy:  0.9802690582959641
classfication_report               precision    recall  f1-score   support

         ham       0.99      0.98      0.99       965
        spam       0.91      0.95      0.93       150

    accuracy                           0.98      1115
   macro avg       0.95      0.97      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [33]:
#upgrade model
#Use TF-IDF better than bag of words

from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 3000, ngram_range = (1,2))
xtf = tf.fit_transform(df['cleaned_message'])

In [34]:
xtf.shape

(5572, 3000)

In [35]:
from sklearn.model_selection import train_test_split

ytf = df['Category']
xtf_train, xtf_test, ytf_train, ytf_test = train_test_split(xtf,ytf, test_size = 0.2, random_state = 42)

In [36]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(xtf_train, ytf_train)

LogisticRegression()

In [37]:
ytf_pred = lr.predict(xtf_test)

In [38]:
from sklearn.metrics import accuracy_score, confusion_matrix
print("Accuracy: ", accuracy_score(ytf_test, ytf_pred))
print(confusion_matrix(ytf_test, ytf_pred))

Accuracy:  0.9605381165919282
[[964   1]
 [ 43 107]]


In [39]:
def predict_nb(text):
    processed = preprocess(text)
    vector = tf.transform([processed])
    return NB.predict(vector)[0]

def predict_lr(text):
    processed = preprocess(text)
    vector = tf.transform([processed])
    return lr.predict(vector)[0]

In [40]:
predict_nb("Congratulations! You won a free ticket")


ValueError: X has 3000 features, but MultinomialNB is expecting 8069 features as input.

In [ ]:
predict_lr("Hey, are we meeting today?")